# Probe training on merged shards

Same logic as `train.ipynb`'s probe-training section. Only difference: `X` and `y` are loaded from the `merged shards (output of `shard_merge.ipynb`) instead of being produced in-memory by a generation loop.

Runs on CPU in a few minutes. No GPU, no compute units.

In [ ]:
# === Colab bootstrap ===
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/cs639-outputs"
else:
    OUTPUT_DIR = "."

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"IN_COLAB={IN_COLAB}  OUTPUT_DIR={OUTPUT_DIR}")

In [ ]:
%pip install -q torch pandas h5py scikit-learn

## Config

In [ ]:
# --- Inputs (from shard_merge.ipynb) ---
HDF5_PATH    = os.path.join(OUTPUT_DIR, "traces_all.h5")

# --- Outputs ---
PROBE_PATH   = os.path.join(OUTPUT_DIR, "linear_probe.pt")
METRICS_PATH = os.path.join(OUTPUT_DIR, "probe_metrics.json")

# --- Hyperparameters ---
HIDDEN_DIM   = 3584   # DeepSeek-R1-Distill-Qwen-7B
PROBE_EPOCHS = 30
PROBE_LR     = 5e-4
BATCH_SIZE   = 256
SEED         = 0

# --- Split ratios (problem-level) ---
TRAIN_FRAC   = 0.70
VAL_FRAC     = 0.15
# TEST_FRAC implied: 1 - TRAIN_FRAC - VAL_FRAC

In [ ]:
# === Load X, y, pids from merged HDF5 ===
import torch
import h5py

assert os.path.exists(HDF5_PATH), f"Missing {HDF5_PATH} — run shard_merge.ipynb first"

X_parts, y_parts, pid_parts = [], [], []
with h5py.File(HDF5_PATH, "r") as f:
    for key in sorted(f.keys()):
        if not key.startswith("problem_"):
            continue
        grp = f[key]
        hs = torch.from_numpy(grp["hidden_states"][...]).float()
        if hs.shape[0] == 0:
            continue
        label = int(grp["label"][()])
        pid   = int(key.split("_", 1)[1])
        X_parts.append(hs)
        y_parts.extend([label] * hs.shape[0])
        pid_parts.extend([pid] * hs.shape[0])

X    = torch.cat(X_parts, dim=0)
y    = torch.tensor(y_parts,   dtype=torch.float32)
pids = torch.tensor(pid_parts, dtype=torch.long)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"X: {tuple(X.shape)}  y: {tuple(y.shape)}  pids: {tuple(pids.shape)}  device: {device}")
print(f"Unique problems: {pids.unique().numel()}")
print(f"Label distribution: {int(y.sum())} correct / {int((1 - y).sum())} incorrect")

In [ ]:
# === Group-aware 70/15/15 split by problem ===
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(SEED)

unique_pids = pids.unique()
shuffled    = unique_pids[torch.randperm(unique_pids.numel())]

n_problems = shuffled.numel()
n_train_p  = int(TRAIN_FRAC * n_problems)
n_val_p    = int(VAL_FRAC   * n_problems)
train_pids = shuffled[:n_train_p]
val_pids   = shuffled[n_train_p:n_train_p + n_val_p]
test_pids  = shuffled[n_train_p + n_val_p:]

def mask_of(target_pids):
    return torch.isin(pids, target_pids)

train_mask, val_mask, test_mask = mask_of(train_pids), mask_of(val_pids), mask_of(test_pids)

# Sanity: splits are disjoint and cover everything
assert (train_mask & val_mask).sum() == 0
assert (train_mask & test_mask).sum() == 0
assert (val_mask & test_mask).sum() == 0
assert (train_mask | val_mask | test_mask).sum() == len(pids)

train_ds = TensorDataset(X[train_mask], y[train_mask])
val_ds   = TensorDataset(X[val_mask],   y[val_mask])
test_ds  = TensorDataset(X[test_mask],  y[test_mask])

loader      = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader  = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds,  batch_size=BATCH_SIZE)

def label_balance(yy):
    pos = int(yy.sum().item())
    return f"{pos}/{len(yy)} pos ({pos / max(len(yy), 1):.2%})"

print(f"Train: {len(train_pids)} problems, {int(train_mask.sum())} positions, {label_balance(y[train_mask])}")
print(f"Val:   {len(val_pids)} problems, {int(val_mask.sum())} positions, {label_balance(y[val_mask])}")
print(f"Test:  {len(test_pids)} problems, {int(test_mask.sum())} positions, {label_balance(y[test_mask])}")

In [ ]:
# === Probe, loss, optimizer (same as train.ipynb) ===
import torch.nn as nn

class LinearProbe(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        return self.fc(x).squeeze(-1)

    def predict_proba(self, x):
        return torch.sigmoid(self.forward(x))

probe     = LinearProbe(HIDDEN_DIM).to(device)
optimizer = torch.optim.Adam(probe.parameters(), lr=PROBE_LR)
criterion = nn.BCEWithLogitsLoss()
print(probe)

In [ ]:
# === Training loop with val ROC-AUC checkpointing ===
from sklearn.metrics import roc_auc_score

best_val_auc = 0.0

def collect_probs(model, dl):
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for xb, yb in dl:
            xb = xb.to(device)
            probs.append(torch.sigmoid(model(xb)).cpu())
            labels.append(yb)
    return torch.cat(probs), torch.cat(labels)

for epoch in range(1, PROBE_EPOCHS + 1):
    probe.train()
    total_loss, n_correct, n_total = 0.0, 0, 0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = probe(xb)
        loss   = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(yb)
        n_correct  += ((logits > 0).long() == yb.long()).sum().item()
        n_total    += len(yb)

    val_probs, val_labels = collect_probs(probe, val_loader)
    val_acc  = ((val_probs > 0.5).long() == val_labels.long()).float().mean().item()
    val_auc  = roc_auc_score(val_labels.numpy(), val_probs.numpy())

    saved = ""
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(
            {"state_dict": probe.state_dict(), "hidden_dim": HIDDEN_DIM, "best_val_auc": best_val_auc},
            PROBE_PATH,
        )
        saved = "  ↑ saved"

    print(f"Epoch {epoch:3d}/{PROBE_EPOCHS}  "
          f"loss={total_loss / n_total:.4f}  "
          f"acc={n_correct / n_total:.4f}  "
          f"val_acc={val_acc:.4f}  "
          f"val_auc={val_auc:.4f}{saved}")

print(f"\nBest val_auc={best_val_auc:.4f}")

In [ ]:
# === Test-set evaluation: reload best checkpoint, compute proposal metrics ===
import json
import numpy as np
from sklearn.metrics import (
    roc_auc_score, brier_score_loss,
    accuracy_score, precision_score, recall_score, f1_score,
)

ckpt = torch.load(PROBE_PATH, map_location=device)
best_probe = LinearProbe(ckpt["hidden_dim"]).to(device)
best_probe.load_state_dict(ckpt["state_dict"])
best_probe.eval()

test_probs, test_labels = collect_probs(best_probe, test_loader)
p = test_probs.numpy()
t = test_labels.numpy().astype(int)
preds = (p > 0.5).astype(int)

def expected_calibration_error(probs, labels, n_bins=15):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece, n = 0.0, len(probs)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        in_bin = (probs >= lo) & (probs < hi if i < n_bins - 1 else probs <= hi)
        if not in_bin.any():
            continue
        bin_conf = probs[in_bin].mean()
        bin_acc  = labels[in_bin].mean()
        ece += (in_bin.sum() / n) * abs(bin_conf - bin_acc)
    return float(ece)

metrics = {
    "roc_auc":           float(roc_auc_score(t, p)),
    "ece":               expected_calibration_error(p, t, n_bins=15),
    "brier":             float(brier_score_loss(t, p)),
    "accuracy":          float(accuracy_score(t, preds)),
    "precision":         float(precision_score(t, preds, zero_division=0)),
    "recall":            float(recall_score(t, preds, zero_division=0)),
    "macro_f1":          float(f1_score(t, preds, average="macro", zero_division=0)),
    "n_test_problems":   int(test_pids.numel()),
    "n_test_positions":  int(len(t)),
    "best_val_auc":      float(best_val_auc),
}

with open(METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Best probe -> {PROBE_PATH}")
print(f"Metrics    -> {METRICS_PATH}")
for k, v in metrics.items():
    print(f"  {k:18s} {v}")

In [ ]:
# === Smoke-check: load saved checkpoint fresh and verify predict_proba ∈ [0, 1] ===
fresh_ckpt  = torch.load(PROBE_PATH, map_location="cpu")
fresh_probe = LinearProbe(fresh_ckpt["hidden_dim"])
fresh_probe.load_state_dict(fresh_ckpt["state_dict"])
fresh_probe.eval()

with torch.no_grad():
    sample = fresh_probe.predict_proba(X[:8])

assert sample.dtype == torch.float32
assert (sample >= 0).all() and (sample <= 1).all(), f"out of range: {sample}"
print(f"predict_proba(X[:8]) = {sample.tolist()}")
print("OK — checkpoint loads in a fresh model, predict_proba returns floats in [0, 1].")